In [104]:
import pandas as pd

In [105]:
desired_time_format = "%m/%d/%y, %I:%M:%S %p"

In [106]:
# load in all required data
ai_prs = pd.read_parquet('./data/all_pull_request.parquet')
human_prs = pd.read_parquet('./data/human_pull_request.parquet')
all_repos = pd.read_parquet('./data/all_repository.parquet')
comments = pd.read_parquet('./data/pr_comments.parquet')

In [107]:
# combine prs to group all agents
ai_prs = ai_prs.drop(columns='repo_id')
#all_prs = pd.concat([ai_prs, human_prs])
#all_prs = ai_prs
all_prs = human_prs
print(f"Total entries: {len(all_prs)}")

Total entries: 6618


In [112]:
ai_prs.head()

,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_url,html_url
0,3264016139,1688,`metta code` --> `metta clip` and additional p...,Remove unused `root_key` variable to fix ruff ...,Claude_Code,37011,jacklionheart,closed,2025-07-25T18:15:36Z,2025-07-25T19:17:23Z,2025-07-25T19:17:23Z,https://api.github.com/repos/Metta-AI/metta,https://github.com/Metta-AI/metta/pull/1688
1,3264021033,41,feat: Comprehensive ruff error resolution with...,## 🎯 Mission Accomplished: 100% Ruff Error Res...,Claude_Code,131842369,Draco3310,open,2025-07-25T18:17:57Z,None,None,https://api.github.com/repos/Draco3310/Gal-Fri...,https://github.com/Draco3310/Gal-Friday2/pull/41
2,3264042289,1600,Add Evals frontend implementation plan and HTM...,\nCreate comprehensive implementation plan for...,Claude_Code,6766889,justicart,closed,2025-07-25T18:26:15Z,2025-07-25T23:19:14Z,None,https://api.github.com/repos/bolt-foundry/bolt...,https://github.com/bolt-foundry/bolt-foundry/p...
3,3264042318,1601,Add 4 new BfDs components for Evals interface ...,\nPhase 1 component creation for the Evals fro...,Claude_Code,6766889,justicart,closed,2025-07-25T18:26:16Z,2025-07-25T23:19:11Z,None,https://api.github.com/repos/bolt-foundry/bolt...,https://github.com/bolt-foundry/bolt-foundry/p...
4,3264067496,3,🚀 Complete Frontend-Backend API Integration wi...,## 🎯 Summary\n\nThis PR completes the **fronte...,Claude_Code,42357482,twitchyvr,closed,2025-07-25T18:39:14Z,2025-07-25T18:48:47Z,2025-07-25T18:48:47Z,https://api.github.com/repos/twitchyvr/Spaghetti,https://github.com/twitchyvr/Spaghetti/pull/3


In [108]:
all_prs["agent"].unique()

array(['Human'], dtype=object)

In [109]:
# get data for popular repos by filtering star count
print(f"All repos: {len(all_repos)}")

pop_repos = all_repos[all_repos["stars"] >= 500]
print(f"Popular repos: {len(pop_repos)}")

All repos: 116211
Popular repos: 1496


In [110]:
# update prs to only those in popular repositories and closed
all_prs = all_prs[all_prs["repo_url"].isin(pop_repos["url"])]
all_prs = all_prs[~all_prs["closed_at"].isna()]
print(f"Final entries: {len(all_prs)}")

Final entries: 6129


In [111]:
all_prs["repo_url"].nunique()

773

In [96]:
# reconfigure some elements of the data
all_prs = all_prs.rename(columns={'user_id': 'userid'})
all_prs['userid'] = all_prs['userid'].apply(str)

comments = comments.rename(columns={'user_id': 'userid'})
comments['userid'] = comments['userid'].apply(str)

def change_dates(column): #does ISO8601 to the desired time, running twice will throw an error
    return pd.to_datetime(column, format = "ISO8601").dt.strftime(desired_time_format)

all_prs['created_at'] = change_dates(all_prs['created_at'])
all_prs['closed_at'] = change_dates(all_prs['closed_at'])
all_prs['merged_at'] = change_dates(all_prs['merged_at'])
comments['created_at'] = change_dates(comments['created_at'])

In [97]:
comments.head()

,id,pr_id,user,userid,user_type,created_at,body
0,2927293042,3107321792,coderabbitai[bot],136622811,Bot,"06/01/25, 02:15:35 PM",<!-- This is an auto-generated comment: summar...
1,3090154270,3234660269,Fank,1900106,User,"07/18/25, 05:22:34 PM","claude budget reached, development is on hold ..."
2,2848667986,3037457814,wilsonccccc,6146503,User,"05/03/25, 03:12:37 PM","Hi, thanks for sharing and contributing! Pleas..."
3,2719183506,2915198291,cloudflare-workers-and-pages[bot],73139402,Bot,"03/12/25, 09:34:46 PM",## Deploying instructor-py with &nbsp;<a href=...
4,3013998830,3132410695,razimantv,3823215,User,"06/27/25, 06:16:05 PM",Can you rebase it to the current master? Pleas...


In [98]:
def combine_pr_and_comments(prs, comments):
    pr_num_column = "pr_id"
    pr_key_comments = {}
    
    #put all the comments into a dictionary by pr_id
    indexes = list(comments.columns)
    for comment in comments.iterrows():
        comment = comment[1]

        pr_id = comment[pr_num_column]
        if pr_id in pr_key_comments:
            pr_key_comments[pr_id].append(comment)
        else:
            pr_key_comments[pr_id] = [comment]
    
    #add in new columns for comments and num_comments
    prs["num_comments"] = 0
    prs["comments"] = [{} for _ in range(len(prs))] #change type if structure of comments changes
    
    
    #put prs into a new list and add the comments that correspond to the pr
    new_prs = []
    
    for pr in prs.iterrows():
        pr = pr[1]
        pr_comments = pr_key_comments.get(pr["id"], None)
        
        if pr_comments is not None:
            pr["num_comments"] = len(pr_comments)
            pr["comments"] = {i : comment for i, comment in enumerate(pr_comments)} #create a dict with index:comment pairs
        
        new_prs.append(pr)
         
    #return a new dataframe from the prs with comments
    combined_df = pd.DataFrame(new_prs)
    return combined_df

In [99]:
combined_df = combine_pr_and_comments(all_prs, comments)

In [100]:
print(len(comments))
print(sum(combined_df["num_comments"]))

39122
39122


In [90]:
combined_df.head()

,id,number,title,body,agent,userid,user,state,created_at,closed_at,merged_at,repo_url,html_url,num_comments,comments
7,3264428170,464,🚀 Complete 64-Agent System Implementation,# 🚀 Complete 64-Agent System Implementation\n\...,Claude_Code,2934394,ruvnet,closed,"07/25/25, 09:26:37 PM","07/25/25, 09:29:18 PM",NaN,https://api.github.com/repos/ruvnet/claude-flow,https://github.com/ruvnet/claude-flow/pull/464,0,{}
26,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,108661493,weiliu1031,closed,"07/26/25, 02:59:01 AM","07/29/25, 07:01:20 AM",NaN,https://api.github.com/repos/milvus-io/pymilvus,https://github.com/milvus-io/pymilvus/pull/2911,2,"{0: [3121064306, 3264933329, 'sre-ci-robot', '..."
77,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,7475,MSch,closed,"07/26/25, 01:31:19 PM","07/26/25, 01:37:22 PM","07/26/25, 01:37:22 PM",https://api.github.com/repos/steipete/Peekaboo,https://github.com/steipete/Peekaboo/pull/30,0,{}
192,3234102722,318,chore: Convert hive-mind coordination system t...,## Summary\n\nThis PR converts the AI agent co...,Claude_Code,15803865,lanemc,closed,"07/16/25, 01:00:34 AM","07/17/25, 12:49:29 PM",NaN,https://api.github.com/repos/ruvnet/claude-flow,https://github.com/ruvnet/claude-flow/pull/318,0,{}
238,3214555104,16658,Add function signature breaking change detector,<details><summary>&#x1F6E0 DevTools &#x1F6E0</...,Claude_Code,17039389,harupy,closed,"07/09/25, 05:35:26 AM","07/11/25, 05:13:35 AM","07/11/25, 05:13:35 AM",https://api.github.com/repos/mlflow/mlflow,https://github.com/mlflow/mlflow/pull/16658,3,"{0: [3056782342, 3214555104, 'serena-ruan', '8..."


In [57]:
combined_df[combined_df["num_comments"]>0]

,id,number,title,user,userid,state,created_at,closed_at,merged_at,repo_url,html_url,body,agent,num_comments,comments


In [34]:
combined_df.reset_index(drop=True, inplace=True) # .to_json was complaining about indexing so this is a fix

In [35]:
combined_df.to_json("human_prs.json", orient="index", indent=4) #indent is for readability, may increase file size